# పాఠం 08 - బహు-ఏజెంట్ డిజైన్ నమూనా


## సెటప్


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## బహుళ-ఏజెంట్ వ్యవస్థలు ఎందుకు?

ప్రయాణ ప్రణాళిక వంటి వాస్తవ ప్రపంచ పనులకు లాజిస్టిక్స్, స్థానిక జ్ఞానం, బడ్జెటింగ్ మరియు మరిన్ని రకాలు మాతృక నైపుణ్యాలు అవసరం అవుతాయి. ఏకైక ఏజెంట్ అన్నిటిని చేయడానికి ప్రయత్నిస్తే, అది త్వరగా చికాకు కలిగే విధంగా మారుతుంది.

బహుళ-ఏజెంట్ వ్యవస్థలు ఈ సమస్యను **విశేషజ్ఞత** ద్వారా పరిష్కరిస్తాయి: ప్రతి ఏజెంట్ ఒక వ్యవస్థలో ప్రత్యేక నైపుణ్యంపై దృష్టి సారించి, సాధారణ వ్యక్తిపై ఆధారపడితే మెరుగైన ఫలితాలను ఉత్పత్తి చేస్తుంది. ఇవి **వ్యాప్తి సామర్థ్యాన్ని** కూడా మెరుగుపరుస్తాయి — మీరు కొత్త ఏజెంట్లను (ఉదా: విమాన నిపుణుడు, రెస్టారెంట్ విమర్శకుడు) ప్రస్తుత పనిముట్టును మళ్లీ రాస్తుండకుండా జోడించవచ్చు. ఏజెంట్లు ఒక నిర్మితమైన పైప్‌లైన్ ద్వారా కలిపి, ఒకటి నుండి మరొకదానికి సందర్భాన్ని పంపుతాయి.


## ప్రత్యేక ఏజెంట్స్ సృష్టించడం


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## వరుసగా వర్క్‌ఫ్లో నిర్మాణం

`WorkflowBuilder` మీకు ఏజెంట్లను దిశానిర్దేశిత గ్రాఫ్‌గా కనెక్ట్ చేయడానికి అనుమతిస్తుంది. ఇక్కడ మేము ఒక సింపుల్ రెండు దశల పైప్‌లైన్‌ను సృష్టిస్తున్నాము: **TravelPlanner** యాత్రా ప్రణాళికను తయారు చేస్తుంది, ఆ తర్వాత **TravelConcierge** దానికి సమీక్షInsertaర్ చేసి మెరుగులు చేర్చుతుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## వర్క్‌ఫ్లోకి మరిన్ని ఏజెంట్లను జోడించడం

బహుళ ఏజెంట్ నమూనాలో పెద్ద లాభాలలో ఒకటి దాన్ని ఎంత సులభంగా విస్తరించగలమో అనే విషయం. క్రింద మేము ఒక **BudgetReviewer** ఏజంట్‌ని జోడిస్తున్నాము, ఇది ప్రయాణికుడి బడ్జెట్‌ను పరీక్షించి, ఖర్చులు మితిని దాటి పోవచ్చు అని అనుమానించే అంశాలను గుర్తించి, డబ్బు పొదుపు ప్రత్యామ్నాయాలను సూచిస్తుంది. ఇప్పుడు వర్క్‌ఫ్లో మూడు ఏజెంట్లను వరుసగా నడుపుతుంది:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నది:

1. **ప్రత్యేక ఏజెంట్‌లను సృష్టించడం** — ప్రతిదీ దృష్టి పెట్టే పాత్రతో (యోజన, కాంసియర్జ్, బడ్జెట్ సమీక్ష).
2. **`WorkflowBuilder` మరియు `add_edge` ఉపయోగించి ఏజెంట్‌లను వరస చర్యలో క్లీంచడం**.
3. **బహుళ ఏజెంట్ పైప్‌లైన్ నుండి అవుట్‌పుట్‌ను ప్రాసారం చేయడం**, ఏ ఏజెంట్ మాట్లాడుతుందో ట్రాక్ చేయడం.
4. **ఉన్నత ఏజెంట్‌లను మార్చకుండా కొత్త ఏజెంట్‌లను చైన్‌లో చేర్చడం ద్వారా పని ప్రవాహాన్ని విస్తరించడం**.

బహుళ ఏజెంట్ రూపకల్పన నమూనా ప్రతి ఏజెంట్‌ను సాదారణంగా ఉంచుతుంది, కానీ ఏ ఏజెంట్ ఒంటరిగా సాధించలేని సంపూర్ణ, సమీక్షించిన ఫలితాలు అందిస్తుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
